# **CAN MONEY BUY HAPPINESS?**  <span style="font-size: 0.3em;">

## **1. Motivation**

The 2026 Happy City ranking has just been released, and Copenhagen has once again secured first place. Looking at the rest of the list, it’s standing out how much Northern European cities dominate the top 20.

While it’s well known that Northern European countries are generally wealthier than much of the rest of Europe, we—three non-Danish students who moved here from Germany, Greece and Italy—started to question whether these cities are truly the happiest. This led us to a broader question, which became the title of our project: *Can money buy happiness?*

We don’t claim to have a definitive answer. Instead, we have developed our own perspective, and we want users to form their own opinions based on the factors we considered important. We want them to reflect on how different factors may influence happiness and decide for themselves what matters most.

To support this, we selected ten datasets from the Eurostat database that capture both financial conditions and quality of life:

- Gross Domestic Product (in Purchasing Power Standard)
- Mean income
- Median income
- People at risk of poverty
- Inability to keep home warm
- Deaths related to mental and behavioural disorders due to alcohol use
- Deaths from intentional self-harm
- Overall satisfaction rating
- Life satisfaction rating (by income quantile, household composition, and degree of urbanisation)
- Self-perceived health
- Trust in others

## **2. Basic stats**

### Data cleaning and preprocessing

Since we wanted to create a choropleth map of Europe, we decided not to limit our analysis to EU countries only. Instead, we included a broader set of European countries that would allow us to build a complete and visually consistent map without gaps. Based on this, we filtered our data using the following country codes:

['AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES',
 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL',
 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UK']

To create the choropleth maps, we downloaded a shapefile of Europe from the DTU Data website [[4]](https://data.dtu.dk/articles/dataset/Shapefile_of_European_countries/23686383). Since the original file was too large and computationally heavy for our application, we simplified the geometry while preserving sufficiently accurate country boundaries. Additionally, Austria was missing from the original shapefile, so we integrated it separately using another geographic source [[5]](https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_AUT_0.json).

Initially, we aimed to analyze the period from 2018 to 2025 in order to use the most recent data available while keeping the datasets as complete as possible. However, we eventually decided to exclude 2025 because too many datasets contained missing values for that year.

For a small number of sporadic missing values in other datasets, we used interpolation to estimate the missing entries and maintain continuity in the visualizations. In contrast, for the datasets Life Satisfaction Rating and Overall Satisfaction Rating, data for 2019 and 2020 was unavailable. Since 2020 was also heavily affected by the COVID-19 pandemic, we chose not to interpolate these values and instead left them empty, excluding them from the related bar charts.

Similarly, the Death Cases datasets contained many missing values for 2024. Because these datasets represent actual counts of deaths, we considered interpolation inappropriate and potentially misleading. For this reason, we decided not to estimate the missing values and excluded 2024 from the corresponding visualizations.

The raw data for these datasets consisted of the total number of deaths in each country. However, to make comparisons between countries more meaningful, we also collected population data [[3]](https://ec.europa.eu/eurostat/databrowser/view/tps00001/default/table?lang=en) and converted the values into percentages relative to each country’s population size. This allowed us to compare countries more fairly despite their large differences in population.

We also observed that, for several datasets, UK data was largely unavailable after 2018 or 2019, likely due to Brexit-related changes in data collection. To maintain consistency across the analysis, we therefore removed the UK from those datasets.

We also separated some datasets into more specific categories in order to improve the clarity of the analysis. The Death Cases dataset was divided into two independent datasets: one containing deaths due to intentional self-harm, and another containing deaths related to mental and behavioural disorders caused by alcohol use. Similarly, the Mean and Median Income dataset was split into two separate datasets so that each indicator could be analyzed individually.

Finally, after cleaning and filtering the datasets, we merged them into a unified structure to support comparison and correlation analysis. Since the datasets used different scales and units — including percentages, satisfaction ratings, and values expressed in Purchasing Power Standards (PPS) — we standardized the variables before analysis. This allowed us to compare indicators more consistently and explore possible relationships between financial conditions and mental or social well-being.

At the beginning of the project, we also considered including the dataset *Persons reporting a work-related health problem by type of diagnosis* (df7 in the code). Although the topic was relevant to our analysis, a large portion of the data was marked as “low reliability”. To maintain the overall quality and consistency of the project, we decided not to include this dataset in the final analysis.


Some datasets also provided more detailed breakdowns by age and sex. Rather than focusing only on total values, we decided to include a dedicated deeper analysis combining both attributes. To keep the visualization clear and comparable, this analysis was performed by country only and not across different years. The goal was to provide an overall overview for each country while highlighting which male or female age groups stand out compared to the overall male and female averages across countries.

This deeper analysis was applied to all datasets except *GDP*, *Life satisfaction rating*, and *Inability to keep home warm*, since these datasets did not provide data split by age and sex.

To ensure consistency across datasets, we created a common mapping for age classes, as different datasets used slightly different age-group definitions. In some cases, datasets were missing the total value for age categories. To address this, we generated the missing total rows by calculating the mean value for each country, year, and sex combination, to have the same format as the other datasets.

### Data stats

Our exploratory analysis revealed clear geographic and socio-economic patterns across Europe. Financial indicators such as GDP, mean income, and median income generally showed higher GDP and income values, while Southern and Eastern European countries tended to show lower values. Similar geographic patterns also appeared in datasets such as *People at risk of poverty* and *Inability to keep home warm* datasets, where Balkan and Eastern countries often displayed higher percentages.

Compared to financial indicators, mental and social well-being datasets appeared more concentrated, with smaller differences between countries. Indicators such as the ratings about life satisfaction and *Trust in persons* generally remained within relatively narrow ranges, although Northern European countries consistently scored among the highest values.

The analysis also highlighted that some datasets showed much larger variability than others. Economic indicators and poverty-related measures varied significantly across countries, while health and satisfaction indicators were comparatively more stable. In contrast, the Death Cases datasets revealed noticeable outliers for specific countries, particularly in deaths related to alcohol disorders and self-harm (for example the percentage of death due to mental disorders due to alchol for Slovenia and Latvia in 2021).

Across most datasets, the overall patterns remained relatively stable over the selected years. However, missing values became more frequent in recent years, especially for 2024–2025. Additionally, UK data disappeared after 2018 in several mental well-being datasets, likely due to Brexit-related changes in data collection.

From these initial observations, we could already expect some level of correlation between financial conditions and well-being indicators. Countries with higher income and GDP levels often also showed higher satisfaction and trust values, while countries with higher poverty-related indicators tended to report lower well-being scores. However, the differences were not always proportional, suggesting that economic wealth alone may not fully explain perceived happiness and quality of life.

Overall, the exploratory analysis confirmed that both financial and well-being indicators exhibit strong regional trends, supporting our decision to combine geographic visualizations with comparative charts and correlation analysis.

In [40]:
import pandas as pd
import urllib.request
from urllib.request import urlopen
import json
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly, plotly.io as pio
import plotly.express as px
import folium
from folium.plugins import HeatMap

In [61]:
df1 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Death cases for mental health_suicide (hlth_cd_aro__custom)csv.csv')
df1 = df1[['sex', 'age',
           'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
           'Place of residence',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df1=df1[
    df1['Place of residence']
    == 'All deaths reported in the country'
]
df1 = df1.rename(columns={'OBS_VALUE': 'Number of Deaths'})
df1 = df1[(df1['TIME_PERIOD'] >= 2018) & (df1['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df1 = df1[(df1["age"]=="TOTAL") & (df1["sex"]=="T")].reset_index(drop=True)

# We split the dataset into 2, to have separate datasets for each cause of death

df1a = df1[(df1["International Statistical Classification of Diseases and Related Health Problems (ICD-10)"]=="Mental and behavioural disorders due to use of alcohol")].reset_index(drop=True)
df1a = df1a.rename(columns={'Number of Deaths': 'Number of Deaths due to mental disorders due to alcohol'})
df1a = df1a.drop(columns=['International Statistical Classification of Diseases and Related Health Problems (ICD-10)', 'sex', 'age', 'Place of residence'])
df1b= df1[(df1["International Statistical Classification of Diseases and Related Health Problems (ICD-10)"]=="Intentional self-harm")].reset_index(drop=True)
df1b = df1b.rename(columns={'Number of Deaths': 'Number of Deaths due to intentional self-harm'})
df1b = df1b.drop(columns=['International Statistical Classification of Diseases and Related Health Problems (ICD-10)', 'sex', 'age', 'Place of residence'])

In [62]:
# Conversion into percentage of population that died due to mental disorders due to alcohol and intentional self-harm

df_pop=pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Population Data.csv')
df_pop = df_pop[["geo","Geopolitical entity (reporting)","TIME_PERIOD","OBS_VALUE"]]
df_pop = df_pop.rename(columns={"OBS_VALUE": "Population number"})

df1a = df1a.merge(df_pop,on=["geo","Geopolitical entity (reporting)", "TIME_PERIOD"],how="left")
df1a["Population percentage of deaths due to mental disorders due to alcohol [%]"]=df1a["Number of Deaths due to mental disorders due to alcohol"]/df1a["Population number"]
df1a["Population percentage of deaths due to mental disorders due to alcohol [%]"] *= 100

df1b = df1b.merge(df_pop,on=["geo","Geopolitical entity (reporting)", "TIME_PERIOD"],how="left")
df1b["Population percentage of deaths due to intentional self-harm [%]"]=df1b["Number of Deaths due to intentional self-harm"]/df1b["Population number"]
df1b["Population percentage of deaths due to intentional self-harm [%]"] *= 100

In [63]:
df2 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/GDP in PPs (tec00114).csv')
df2 = df2[['geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df2 = df2.rename(columns={'OBS_VALUE': 'GDP [PPS]'})
df2 = df2[(df2['TIME_PERIOD'] >= 2018) & (df2['TIME_PERIOD'] <= 2024)].reset_index(drop=True)

In [64]:
df3 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Life satisfaciton by income quintile, household composition and degree of urbanisation (ilc_pw02).csv')
df3 = df3[['Degree of urbanisation',
           'Income quantile',
           'Household composition',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE',
           'Observation status (Flag) V2 structure']]
df3 = df3.rename(columns={'OBS_VALUE': 'Life Satisfaction Rating [0-10]'})
df3 = df3[(df3['TIME_PERIOD'] >= 2018) & (df3['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df3 = df3[(df3["Degree of urbanisation"]=="Total") & (df3["Income quantile"]=="Total") 
          & (df3["Household composition"]=="Total")].reset_index(drop=True)
df3 = df3.drop(columns=['Degree of urbanisation', 'Income quantile', 'Household composition', 'Observation status (Flag) V2 structure'])

In [65]:
df4 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Mean&Median Income (ilc_di03).csv')
df4 = df4[['age', 'sex', 'unit',
           'Income and living conditions indicator',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df4 = df4[df4['unit'] == 'PPS']
mean_df = df4[
    df4['Income and living conditions indicator'] == 'Mean equivalised net income'].drop(columns=['Income and living conditions indicator'])
median_df = df4[
    df4['Income and living conditions indicator'] == 'Median equivalised net income'].drop(columns=['Income and living conditions indicator'])
df4 = mean_df.merge(
    median_df,
    on=['age', 'sex', 'unit', 'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD'],
    suffixes=('_mean', '_median')
)
df4 = df4.rename(columns={'OBS_VALUE_mean': 'Mean Income', 'OBS_VALUE_median': 'Median Income'})
df4 = df4[(df4['TIME_PERIOD'] >= 2018) & (df4['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df4 = df4[(df4["age"]=="TOTAL") & (df4["sex"]=="T")].reset_index(drop=True)
df4 = df4.drop(columns=['age', 'sex', 'unit'])  

# We split the dataset into 2, to have separate datasets for mean and median income

df4a=df4.drop(columns="Median Income")
df4b=df4.drop(columns="Mean Income")

In [66]:
df5 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Overall satisfaction (ilc_pw01).csv')
df5 = df5[['International Standard Classification of Education (ISCED 2011)',
           'sex', 'age', 
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE',
           'Observation status (Flag) V2 structure']]
df5 = df5.rename(columns={'OBS_VALUE': 'Overall Satisfaction Rating [0-10]'})
df5 = df5[(df5['TIME_PERIOD'] >= 2018) & (df5['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df5 = df5[(df5["sex"]=="T") & (df5["International Standard Classification of Education (ISCED 2011)"]=="All ISCED 2011 levels")].reset_index(drop=True)
df5 = df5.groupby(['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD'], as_index=False)['Overall Satisfaction Rating [0-10]'].mean()

In [67]:
df6 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/People at risk of poverty (ilc_peps01n).csv')
df6 = df6[['sex', 'age', 'unit',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df6 = df6[df6['unit'] == 'PC']
df6 = df6.rename(columns={'OBS_VALUE': 'People at risk of poverty [%]'})
df6 = df6[(df6['TIME_PERIOD'] >= 2018) & (df6['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df6 = df6[(df6["age"]=="TOTAL") & (df6["sex"]=="T")].reset_index(drop=True)
df6 = df6.drop(columns = ['sex', 'age', 'unit'])

In [ ]:
# df7 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Persons reporting a work-related health problem by type of diagnose - stress,anxiety(hsw_pb5__custom).csv')
# df7.columns
# NOT USED ANYMORE, TOO MANY LOW RELIABILITY VALUES

In [68]:
df8 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Self-perceived health (hlth_silc_01).csv')
df8 = df8[['Labour force and employment status', 'age', 'sex', 'Level',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df8 = df8.rename(columns={'OBS_VALUE': 'Self-perceived health [%]'})
df8 = df8[(df8['TIME_PERIOD'] >= 2018) & (df8['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df8 = df8[(df8["sex"]=="T")].reset_index(drop=True)
df8 = df8.groupby(['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD'], as_index=False)['Self-perceived health [%]'].mean()

In [69]:
df9 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Someone to rely on (ilc_pw06).csv')
df9 = df9[['sex', 'age', 
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df9 = df9.rename(columns={'OBS_VALUE': 'Someone to rely on [%]'})
df9 = df9[(df9['TIME_PERIOD'] >= 2018) & (df9['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df9 = df9[(df9["sex"]=="T")].reset_index(drop=True)
df9 = df9.groupby(['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD'], as_index=False)['Someone to rely on [%]'].mean()

In [70]:
df10 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Trust in persons (ilc_pw03).csv')
df10 = df10[['sex', 'age', 
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df10 = df10.rename(columns={'OBS_VALUE': 'Trust in persons [%]'})
df10 = df10[(df10['TIME_PERIOD'] >= 2018) & (df10['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df10 = df10[(df10["sex"]=="T")].reset_index(drop=True)
df10 = df10.groupby(['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD'], as_index=False)['Trust in persons [%]'].mean()

In [71]:
df11 = pd.read_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/Inability to keep home warm (ilc_mdes01).csv')
df11 = df11[['Type of household', 'Income situation in relation to the risk of poverty threshold',
           'geo',
           'Geopolitical entity (reporting)',
           'TIME_PERIOD',
           'OBS_VALUE']]
df11 = df11.rename(columns={'OBS_VALUE': 'Inability to keep home warm [%]'})
df11 = df11[(df11['TIME_PERIOD'] >= 2018) & (df11['TIME_PERIOD'] <= 2024)].reset_index(drop=True)
df11 = df11[ (df11["Income situation in relation to the risk of poverty threshold"]=="Total") 
            & (df11["Type of household"]=="Total")].reset_index(drop=True)
df11 = df11.drop(columns = ['Type of household', 'Income situation in relation to the risk of poverty threshold'])

In [72]:
# Filtering the datasets to keep only the 35 European countries that we are interested in for our analysis, and resetting the index of each dataset
countries = [
    'AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU',
    'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK'
]
df1a = df1a[df1a["geo"].isin(countries)].reset_index(drop=True)
df1b = df1b[df1b["geo"].isin(countries)].reset_index(drop=True)
df2 = df2[df2["geo"].isin(countries)].reset_index(drop=True)
df3 = df3[df3["geo"].isin(countries)].reset_index(drop=True)
df4a = df4a[df4a["geo"].isin(countries)].reset_index(drop=True)
df4b = df4b[df4b["geo"].isin(countries)].reset_index(drop=True)
df5 = df5[df5["geo"].isin(countries)].reset_index(drop=True)
df6 = df6[df6["geo"].isin(countries)].reset_index(drop=True)
df8 = df8[df8["geo"].isin(countries)].reset_index(drop=True)
df9 = df9[df9["geo"].isin(countries)].reset_index(drop=True)
df10 = df10[df10["geo"].isin(countries)].reset_index(drop=True)
df11 = df11[df11["geo"].isin(countries)].reset_index(drop=True)

In [73]:
# * HANDLING MISSING VALUES *

# Italy is missing the value of self-perceived health for 2020, so we will add it and then interpolate it based on the values of all the other years to have a more accurate estimation of the missing value
new_row = {
    "Unnamed: 0": None,
    "geo": "IT",
    "Geopolitical entity (reporting)": "Italy",
    "TIME_PERIOD": 2020,
    "Self-perceived health [%]": np.nan
}
df8 = pd.concat([df8, pd.DataFrame([new_row])], ignore_index=True)
df8 = df8.sort_values(["geo", "TIME_PERIOD"])
df8.loc[df8["geo"] == "IT", "Self-perceived health [%]"] = (
    df8[df8["geo"] == "IT"]["Self-perceived health [%]"]
    .interpolate()
)

# Albania is missing the value of life satisfaction for 2023, so we will add it and then interpolate it based on the values of all the other years to have a more accurate estimation of the missing value
new_row = {
    "Unnamed: 0": None,
    "geo": "AL",
    "Geopolitical entity (reporting)": "Albania",
    "TIME_PERIOD": 2023,
    "Life Satisfaction Rating [0-10]": np.nan
}
df3 = pd.concat([df3, pd.DataFrame([new_row])], ignore_index=True)
df3 = df3.sort_values(["geo", "TIME_PERIOD"])
df3.loc[df3["geo"] == "AL", "Life Satisfaction Rating [0-10]"] = (
    df3[df3["geo"] == "AL"]["Life Satisfaction Rating [0-10]"]
    .interpolate()
)

# For the other missing values, we have NaNs so we don't need to add new rows
# Then again we will interpolate them based on the values of all the other years to have a more accurate estimation of the missing values

df3 = df3.sort_values(["geo", "TIME_PERIOD"])
df3["Life Satisfaction Rating [0-10]"] = (
    df3.groupby("geo")["Life Satisfaction Rating [0-10]"]
       .transform(lambda x: x.interpolate())
)

df5 = df5.sort_values(["geo", "TIME_PERIOD"])
df5["Overall Satisfaction Rating [0-10]"] = (
    df5.groupby("geo")["Overall Satisfaction Rating [0-10]"]
       .transform(lambda x: x.interpolate())
)

df10 = df10.sort_values(["geo", "TIME_PERIOD"])
df10["Trust in persons (%)"] = (
    df10.groupby("geo")["Trust in persons [%]"]
       .transform(lambda x: x.interpolate())
)

In [ ]:
# Saving the cleaned datasets as csv files

# df1a.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df1a.csv')
# df1b.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df1b.csv')
# df2.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df2.csv')
# df3.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df3.csv')
# df4a.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df4a.csv')
# df4b.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df4b.csv')
# df5.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df5.csv')
# df6.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df6.csv')
# df8.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df8.csv')
# df9.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df9.csv')
# df10.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df10.csv')
# df11.to_csv('/Users/giorgiasantinelli/Documents/MONICA/Università/DTU/Social_data_analysis/Final project/datasets/df11.csv')

## **3. Data analysis**

The choropleth maps highlighted the regional differences very clearly. For example, countries such as Ireland, Luxembourg, Norway, and Switzerland consistently appeared among the highest values for economic indicators, whereas Balkan and Eastern European countries generally showed lower values. At the same time, the patterns across years remained relatively stable, suggesting that regional differences were stronger than yearly fluctuations during the selected period.

The bar charts used for mental and social well-being indicators also revealed noticeable differences between countries. Northern European countries generally showed higher values in trust, life satisfaction, and overall satisfaction ratings, while countries with lower economic indicators often reported lower satisfaction levels as well. However, some exceptions emerged, suggesting that financial wealth alone does not fully explain happiness and well-being.

To explore the relationship between financial conditions and well-being, we analyzed correlations between the financial indicators and the mental or social datasets using scatter plots and regression lines.
- GDP, mean income, and median income generally showed positive correlations with Life Satisfaction, Overall Satisfaction, and Self-perceived Health. Countries with stronger economic indicators therefore tended to report higher levels of perceived well-being, with satisfaction-related indicators showing the clearest trends.
- In contrast, People at Risk of Poverty and Inability to Keep Home Warm showed negative correlations with satisfaction and health measures. Countries with higher poverty-related percentages generally reported lower well-being scores.
- The relationship with Trust in Persons appeared weaker and more scattered, suggesting that social trust may depend on additional cultural or social factors beyond economic conditions alone.
- The death-related datasets produced less clear results. Intentional self-harm showed slight negative correlations with financial indicators, while alcohol-related mental disorder deaths appeared more irregular and influenced by additional factors.

Overall, the analysis suggests that financial stability is positively related to well-being, but, again, economic wealth alone does not fully explain happiness and quality of life across countries.

## **4. Genre**

Genre: Partitioned Poster and Slide Show (Segel & Heer (2010) — §4.3​)
- Keep the choropleth map as the visual anchor => geographic intuition first​
- Supporting charts appear alongside without burying the main message​
- Reader-driven exploration: wealth metrics & time slider => author sets the frame, user explores​
- Avoid overwhelming scrolling; all key indicators visible in one view​

Visual structuring as Visual Narrative 
Interactivity as Narrative Structure

**Genre**

We adopted a hybrid genre combining *Partitioned Poster* and *Slide Show* [Segel & Heer (2010) — §4.3​].

The *Partitioned Poster* aspect is reflected in our layout: multiple visual elements (maps and charts) are presented simultaneously within a single view. This allows users to compare financial and well-being indicators without losing context, while maintaining a clear visual hierarchy. The choropleth map acts as the main visual anchor, using geographic familiarity as an intuitive entry point into the analysis.

At the same time, we incorporate elements of a *Slide Show* through the use of interactive tabs, filters, and time selection controls. These introduce a temporal and thematic progression, guiding users through different datasets and perspectives without enforcing a fixed linear narrative.

This hybrid approach was chosen to balance author-driven structure and reader-driven exploration. Rather than presenting a single conclusion, we provide a structured visual environment where users can explore relationships between wealth and well-being at their own pace, which, in our opinion, aligns with the exploratory nature of our research question.

**Visual Narrative**

We primarily relied on visual structuring techniques to support the narrative.

- Consistent visual platform: all views share the same layout, colors, and interaction logic. This helps users stay oriented when switching between indicators, years, or demographic views. As highlighted in the paper, consistent structure helps users “identify their position within the visualization”.
- Establishing shot: the choropleth map serves as the main entry point, immediately framing the geographic context. This follows the idea that visual narratives benefit from a clear starting point that introduces the “scene” to the user .
- Feature distinction and highlighting: colors are used consistently to distinguish European regions and emphasize differences between countries directing attention to spatial patterns (e.g., North vs South Europe). Scatter plots also highlight trends and outliers through regression lines.
- Limited transition guidance: instead of complex animations, we use simple updates (e.g., time slider changes, tabs, filters, synchronized plot updates) to maintain continuity without overwhelming the user. This should avoid disorientation while keeping the interaction lightweight.

We intentionally avoided overly complex visual effects to keep the focus on comparison, interpretation, and exploration, which is central to our project.

**Narrative Structure**

We mainly used interactivity-driven narrative structure, complemented by light ordering and messaging.

- Interactivity:    
    - Filtering and selection (datasets, time slider): Users can explore different indicators and time periods, enabling them to test their own hypotheses about happiness and wealth.
    - User-directed path: There is no fixed sequence; users decide what to explore and in which order. This aligns with a reader-driven approach, where exploration supports personal interpretation.
    - Stimulating default view: The initial map provides an immediate, meaningful overview that encourages further exploration, similar to a “visual lead” described in the paper.

    Since our goal is not to provide a single answer but to let users form their own opinion, interactivity is essential. It transforms the visualization from a static explanation into an exploratory tool.

- Ordering:      
    although the experience is non-linear, the interface still provides implicit guidance through the organization of the views: maps first for geographic overview, followed by comparative charts and correlation plots for deeper analysis.

- Messaging:      
    we intentionally kept annotations and explanations minimal. Instead of explicitly stating conclusions, the visualizations themselves communicate the main patterns and encourage users to reflect on the relationship between financial conditions and well-being.

## **5. Visualizations**

We chose a combination of choropleth maps, bar charts, and scatter plots to represent the different aspects of our data and support the main question of the project: *Can money buy happiness?*

The choropleth maps are used to display financial indicators across Europe. Since our analysis is strongly connected to geography and regional differences, maps provide an immediate and intuitive way to identify patterns between countries. They allow users to quickly compare Northern, Southern, Eastern, and Western European regions and observe how wealth-related indicators are distributed spatially.

Alongside the maps, we used bar charts to represent mental and social well-being indicators. Bar charts make it easier to compare exact values between countries and highlight differences that may be less visible on a map. Placing these charts next to the choropleth maps also encourages users to visually relate financial and well-being indicators at the same time.

We also included scatter plots to explore possible correlations between financial variables and mental or social indicators. These visualizations are particularly useful for investigating whether higher wealth is associated with higher happiness, trust, health, or life satisfaction. Rather than directly answering the question, the scatter plots support exploration and interpretation, allowing users to identify trends, clusters, and outliers on their own.

For the deeper analysis by age and sex, we created interactive line plots, combined with bars representing total values, that allow users to dynamically select different age classes and indicators. The visualization is divided into three synchronized subplots representing male, female, and total values, making comparisons between demographic groups more immediate. Financial-related indicators are displayed on one side of the interface, while mental and social well-being indicators are shown on the other.

Both the choropleth maps and the age-and-sex analysis section include interactive tabs that allow users to choose which indicators to visualize and compare. This supports a more flexible and exploratory analysis of demographic and geographic differences across countries.

Together, these visualizations balance overview and detail while supporting both comparison and exploration, which, in our opinion, fits the open and exploratory nature of our project.

### Choropleth map -- financial related plots

In [17]:
gdf = gpd.read_file("Europe_merged.shp")
# print(gdf.head())
# print(gdf.crs)
# print(gdf.columns)
# gdf.plot()
gdf = gdf.to_crs("EPSG:4326")
iso3_to_iso2 = {
    'ALB': 'AL', 'AUT': 'AT', 'BEL': 'BE', 'BGR': 'BG', 'CHE': 'CH',
    'CYP': 'CY', 'CZE': 'CZ', 'DEU': 'DE', 'DNK': 'DK', 'EST': 'EE',
    'GRC': 'EL', 'ESP': 'ES', 'FIN': 'FI', 'FRA': 'FR', 'HRV': 'HR',
    'HUN': 'HU', 'IRL': 'IE', 'ITA': 'IT', 'LTU': 'LT', 'LUX': 'LU',
    'LVA': 'LV', 'MKD': 'MK', 'MLT': 'MT', 'NLD': 'NL', 'NOR': 'NO',
    'POL': 'PL', 'PRT': 'PT', 'ROU': 'RO', 'SRB': 'RS', 'SWE': 'SE',
    'SVN': 'SI', 'SVK': 'SK', 'GBR': 'UK', 'XKO': 'XK', 'ISL': 'IS',   
    'TUR': 'TR', 'BIH': 'BA', 'MNE': 'ME' 
    }
gdf["GID_0"] = gdf["GID_0"].map(iso3_to_iso2)
# print(gdf[gdf["GID_0"].isna()])
# print(gdf.head())

filtered = gdf[gdf["GID_0"].isin(countries)]
filtered["geometry"] = filtered["geometry"].simplify(0.01, preserve_topology=True)
filtered.to_file("europe_selected_countries.geojson", driver="GeoJSON")


with open("europe_selected_countries.geojson", "r", encoding="utf-8") as f:
    countries_geojson = json.load(f)

In [18]:
import requests
import json

# high-detail Austria boundary from GADM
url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_AUT_0.json"

austria_geojson = requests.get(url).json()

# take Austria feature
austria_feature = austria_geojson["features"][0]

# rewrite properties to match your file
austria_feature["properties"] = {
    "GID_0": "AT",
    "COUNTRY": "Austria"
}

# load your existing GeoJSON
with open("europe_selected_countries.geojson", "r", encoding="utf-8") as f:
    countries_geojson = json.load(f)

# remove Austria if accidentally already present
countries_geojson["features"] = [
    f for f in countries_geojson["features"]
    if f["properties"].get("GID_0") != "AT"
]

# add Austria
countries_geojson["features"].append(austria_feature)

# save
with open("countries_with_austria.geojson", "w", encoding="utf-8") as f:
    json.dump(countries_geojson, f)

In [ ]:
years = sorted(df2["TIME_PERIOD"].unique())

for year in years:
    df_year = df2[df2["TIME_PERIOD"] == year]

    fig = px.choropleth_map(
        df_year,
        geojson=countries_geojson,
        locations="geo",
        featureidkey="properties.GID_0",
        color="GDP [PPS]",
        color_continuous_scale="Viridis",
        range_color=(min(df_year["GDP [PPS]"]), max(df_year["GDP [PPS]"])),
        map_style="carto-positron",
        zoom=2,
        center={"lat": 55, "lon": 15},
        opacity=0.5,
        labels={"GDP [PPS]": "[PPS]"}
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    filename = f"GDP_{year}.html"
    plotly.offline.plot(fig, filename=filename, auto_open=False)
    fig.write_html(filename, include_plotlyjs='cdn')

In [65]:
years = sorted(df4["TIME_PERIOD"].unique())

for year in years:
    df_year = df4[df4["TIME_PERIOD"] == year]

    fig = px.choropleth_map(
        df_year,
        geojson=countries_geojson,
        locations="geo",
        featureidkey="properties.GID_0",
        color="Mean Income",
        color_continuous_scale="Viridis",
        range_color=(min(df_year["Mean Income"]), max(df_year["Mean Income"])),
        map_style="carto-positron",
        zoom=2,
        center={"lat": 55, "lon": 15},
        opacity=0.5,
        labels={"Mean Income": "[PPS]"}
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    filename = f"Mean Income_{year}.html"
    plotly.offline.plot(fig, filename=filename, auto_open=False)
    fig.write_html(filename, include_plotlyjs='cdn')

In [63]:
years = sorted(df4["TIME_PERIOD"].unique())

for year in years:
    df_year = df4[df4["TIME_PERIOD"] == year]

    fig = px.choropleth_map(
        df_year,
        geojson=countries_geojson,
        locations="geo",
        featureidkey="properties.GID_0",
        color="Median Income",
        color_continuous_scale="Viridis",
        range_color=(min(df_year["Median Income"]), max(df_year["Median Income"])),
        map_style="carto-positron",
        zoom=2,
        center={"lat": 55, "lon": 15},
        opacity=0.5,
        labels={"Median Income": "[PPS]"}
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    filename = f"Median Income_{year}.html"
    plotly.offline.plot(fig, filename=filename, auto_open=False)
    fig.write_html(filename, include_plotlyjs='cdn')

In [ ]:
years = sorted(df6["TIME_PERIOD"].unique())

for year in years:
    df_year = df6[df6["TIME_PERIOD"] == year]

    fig = px.choropleth_map(
        df_year,
        geojson=countries_geojson,
        locations="geo",
        featureidkey="properties.GID_0",
        color="People at risk of poverty [%]",
        color_continuous_scale="Viridis",
        range_color=(min(df_year["People at risk of poverty [%]"]), max(df_year["People at risk of poverty [%]"])),
        map_style="carto-positron",
        zoom=2,
        center={"lat": 55, "lon": 15},
        opacity=0.5,
        labels={"People at risk of poverty [%]": "[%]"}
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    filename = f"People at risk of poverty_{year}.html"
    plotly.offline.plot(fig, filename=filename, auto_open=False)
    fig.write_html(filename, include_plotlyjs='cdn')

In [ ]:
years = sorted(df11["TIME_PERIOD"].unique())

for year in years:
    df_year = df11[df11["TIME_PERIOD"] == year]

    fig = px.choropleth_map(
        df_year,
        geojson=countries_geojson,
        locations="geo",
        featureidkey="properties.GID_0",
        color="Inability to keep home warm [%]",
        color_continuous_scale="Viridis",
        range_color=(min(df_year["Inability to keep home warm [%]"]), max(df_year["Inability to keep home warm [%]"])),
        map_style="carto-positron",
        zoom=2,
        center={"lat": 55, "lon": 15},
        opacity=0.5,
        labels={"Inability to keep home warm [%]": "[%]"}
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    filename = f"Inability to keep home warm_{year}.html"
    plotly.offline.plot(fig, filename=filename, auto_open=False)
    fig.write_html(filename, include_plotlyjs='cdn')

### Bar charts -- mental and well-being related plots

In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import colorsys
from collections import defaultdict

# DataFrames in plot order
dfs = [df1a, df1b, df3, df5, df8, df10]

# Value columns in the same order as dfs
value_cols = [
    "Population percentage of deaths due to mental disorders due to alcohol [%]",          # df1a
    "Population percentage of deaths due to intentional self-harm [%]",        # df1b
    "Life Satisfaction Rating [0-10]",    # df3
    "Overall Satisfaction Rating [0-10]", # df5
    "Self-perceived health [%]",          # df8
    "Trust in persons [%]"                # df10
]
dfs = [df[df["geo"] != "UK"] for df in dfs]

# All countries
all_countries = sorted(set().union(*[
    df["Geopolitical entity (reporting)"].dropna().unique()
    for df in dfs
]))

# Colours
region_map = {
    # Northern Europe
    "Denmark": "Northern Europe",
    "Estonia": "Northern Europe",
    "Finland": "Northern Europe",
    "Ireland": "Northern Europe",
    "Lithuania": "Northern Europe",
    "Latvia": "Northern Europe",
    "Norway": "Northern Europe",
    "Sweden": "Northern Europe",
    "United Kingdom": "Northern Europe",

    # Western Europe
    "Austria": "Western Europe",
    "Belgium": "Western Europe",
    "Switzerland": "Western Europe",
    "Germany": "Western Europe",
    "France": "Western Europe",
    "Luxembourg": "Western Europe",
    "Netherlands": "Western Europe",

    # Southern Europe
    "Albania": "Southern Europe",
    "Bosnia and Herzegovina": "Southern Europe",  
    "Montenegro": "Southern Europe",              
    "Cyprus": "Southern Europe",
    "Greece": "Southern Europe",
    "Spain": "Southern Europe",
    "Croatia": "Southern Europe",
    "Italy": "Southern Europe",
    "Malta": "Southern Europe",
    "Portugal": "Southern Europe",
    "Slovenia": "Southern Europe",
    "Serbia": "Southern Europe",
    "North Macedonia": "Southern Europe",

    # Eastern Europe
    "Bulgaria": "Eastern Europe",
    "Czechia": "Eastern Europe",
    "Hungary": "Eastern Europe",
    "Poland": "Eastern Europe",
    "Romania": "Eastern Europe",
    "Slovakia": "Eastern Europe",
}

region_colors = {
    "Western Europe": "#5fa8d3",  # blue
    "Eastern Europe": "#7e57c2",  # purple
    "Northern Europe": "#6fbf73",  # green
    "Southern Europe": "#ee964b",  # orange
}

def adjust_lightness(hex_color, factor):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4))
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})"

region_groups = defaultdict(list)

for country in all_countries:
    region = region_map.get(country, "Other")
    region_groups[region].append(country)

color_map = {}

for region, countries_in_region in region_groups.items():
    countries_in_region = sorted(countries_in_region)
    base_color = region_colors.get(region, "#7f7f7f")
    n = len(countries_in_region)

    for i, country in enumerate(countries_in_region):
        factor = 0.7 + 0.6 * (i / max(1, n - 1))
        color_map[country] = adjust_lightness(base_color, factor)

# Plot
for year in range(2018, 2025):
    
    valid_plots = []

    for df_plot, value_col in zip(dfs, value_cols):
        df_year = df_plot[df_plot["TIME_PERIOD"] == year]

        if len(df_year) > 10:
            valid_plots.append((df_plot, value_col))

    if len(valid_plots) == 0:
        continue

    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=[value_col for _, value_col in valid_plots],
        shared_yaxes=False,
        shared_xaxes=False,
        vertical_spacing=0.1
    )

    for country in all_countries:

        for i, (df_plot, value_col) in enumerate(valid_plots):
            row = i // 2 + 1
            col = i % 2 + 1

            df_year = df_plot[df_plot["TIME_PERIOD"] == year]

            country_df = df_year[
                df_year["Geopolitical entity (reporting)"] == country
            ]

            if country_df.empty:
                x_values = [country]
                y_values = [None]
            else:
                x_values = country_df["Geopolitical entity (reporting)"]
                y_values = country_df[value_col]

            fig.add_trace(
                go.Bar(
                    x=x_values,
                    y=y_values,
                    name=country,
                    visible="legendonly",
                    legendgroup=country,
                    showlegend=(i == 0),
                    marker=dict(
                        color=color_map[country],
                        line=dict(width=0)
                    )
                ),
                row=row,
                col=col
            )

    # # Scales
    # for col in [1, 2]:
    #     fig.update_yaxes(range=[0, 0.025], row=1, col=col)

    # for col in [1, 2]:
    #     fig.update_yaxes(range=[0, 10], row=2, col=col)

    # for col in [1, 2]:
    #     fig.update_yaxes(range=[0, 50], row=3, col=col)

    fig.update_layout(
        height=750,
        width=1200,
        barmode="group",
        margin=dict(t=20, r=220, b=80, l=60),
        title=None,
        font=dict(size=9),
        legend=dict(
            title="Countries",
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top",
            traceorder="normal",
            groupclick="togglegroup"
        )
    )

    fig.update_xaxes(
        type="category",
        tickangle=45,
        title_font=dict(size=9),
        tickfont=dict(size=7)
    )

    fig.update_yaxes(
        title_text="Value",
        title_font=dict(size=9),
        tickfont=dict(size=8)
    )

    # X-axis titles only in last row
    for col in [1, 2]:
        fig.update_xaxes(title_text="", row=1, col=col)
        fig.update_xaxes(title_text="", row=2, col=col)
        fig.update_xaxes(title_text="Geopolitical entity", row=3, col=col)

    for annotation in fig.layout.annotations:
        annotation.font = dict(size=9)

    filename = f"Mental_health_{year}.html"
    fig.write_html(filename, include_plotlyjs="cdn")

### Scatter plots -- plots about correlations between financial key and mental and well-being 

In [75]:
# * MERGING THE DATASETS *

from functools import reduce

keys = ["geo", "Geopolitical entity (reporting)", "TIME_PERIOD"]

dfs = [df1a, df1b, df2, df3, df4a, df4b, df5, df6, df8, df10, df11]

value_cols = [
    "Population percentage of deaths due to mental disorders due to alcohol [%]",
    "Population percentage of deaths due to intentional self-harm [%]",
    'GDP [PPS]',
    'Life Satisfaction Rating [0-10]',
    'Mean Income', 'Median Income',
    "Overall Satisfaction Rating [0-10]",
    'People at risk of poverty [%]',
    "Self-perceived health [%]",
    "Trust in persons [%]",
    'Inability to keep home warm [%]'
]

# keep only keys + relevant value column from each dataframe
dfs_clean = []

for df, value_col in zip(dfs, value_cols):
    temp = df[keys + [value_col]].copy()
    
    # avoid duplicate rows per country-year
    temp = temp.drop_duplicates(subset=keys)
    
    dfs_clean.append(temp)

# merge all datasets into one wide dataset
df_merged = reduce(
    lambda left, right: left.merge(right, on=keys, how="outer"),
    dfs_clean
)

df_clean=df_merged.dropna()
df_clean=df_clean.drop(columns="TIME_PERIOD")

In [76]:
# * STANDARDIZING THE DATA *
import sklearn.preprocessing
from sklearn.preprocessing import StandardScaler

features = df_clean.drop(columns=[
    "geo",
    "Geopolitical entity (reporting)"
])

scaler = StandardScaler()
df_scaled = scaler.fit_transform(features)

df_scaled = pd.DataFrame(df_scaled, columns=features.columns)

In [79]:
mental_vars = [
    "Population percentage of deaths due to mental disorders due to alcohol [%]",
    "Population percentage of deaths due to intentional self-harm [%]",
    "Life Satisfaction Rating [0-10]",
    "Overall Satisfaction Rating [0-10]",
    "Self-perceived health [%]",
    "Trust in persons [%]"
]

financial_vars = [
    "GDP [PPS]",
    "Mean Income",
    "Median Income",
    "People at risk of poverty [%]",
    "Inability to keep home warm [%]"
]

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import re

def clean_label(text):
    return re.sub(r"\s*\[.*?\]", "", text)

for financial in financial_vars:

    valid_plots = []

    # check which mental variables have enough data for this financial variable
    for mental in mental_vars:
        df_plot = df_scaled[[financial, mental]].dropna()

        if len(df_plot) > 10:
            valid_plots.append((mental, df_plot))

    if len(valid_plots) == 0:
        continue

    # Subplot 
    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=None,
        shared_yaxes=False
    )

    for i, (mental, df_plot) in enumerate(valid_plots, start=1):

        row = 1 if i <= 3 else 2
        col = i if i <= 3 else i - 3

        x = df_plot[financial]
        y = df_plot[mental]

        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode='markers',
                opacity=0.6,
                showlegend=False
            ),
            row=row,
            col=col
        )

        slope, intercept = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = slope * x_line + intercept

        fig.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode='lines',
                line=dict(color='red'),
                showlegend=False
            ),
            row=row,
            col=col
        )

    fig.update_layout(
        title=None,
        width=1200,
        height=800,
        font=dict(size=9),
        margin=dict(t=50)
    )

    for i, (mental, _) in enumerate(valid_plots, start=1):

        row = 1 if i <= 3 else 2
        col = i if i <= 3 else i - 3

        words = mental.split()
        lines = []
        line = ""

        for word in words:
            if len(line) + len(word) < 25:
                line += " " + word if line else word
            else:
                lines.append(line)
                line = word

        lines.append(line)

        wrapped_text = "<br>".join(lines)
        wrapped_text = clean_label(wrapped_text)

        fig.update_xaxes(
            title_text=wrapped_text,
            title_font=dict(size=9),
            row=row,
            col=col
        )

    for i in range(1, len(valid_plots) + 1):

        row = 1 if i <= 3 else 2
        col = i if i <= 3 else i - 3

        fig.update_yaxes(
            title_text=clean_label(financial) if col == 1 else "",
            title_font=dict(size=9),
            row=row,
            col=col
        )

    # fig.show()
    
    filename = f"Correlations_{financial}.html"
    fig.write_html(filename)

## **6. Discussion**

Overall, we believe the project successfully highlights meaningful relationships between financial conditions and well-being indicators across Europe. One of the strongest aspects of the project was the combination of multiple datasets covering both economic and mental or social dimensions. We also dedicated significant effort to data cleaning and preprocessing, which was one of the most challenging parts of the work due to the large amount of data and missing, inconsistent, or unreliable data across datasets.

At the same time, we are aware that the project still has limitations and could be improved in several ways.

One important limitation concerns data availability over time. Initially, we planned to include data up to 2025 in order to use the most recent information available, but the amount of missing data made this impossible. We also considered reducing the temporal granularity by analyzing only selected years (for example every three years starting from 2018). However, this would have reduced the amount of usable data even further, so we ultimately decided to keep yearly data despite some incomplete years. We recognize that this choice may affect the consistency of some comparisons.

Another limitation concerns the range of indicators included in the analysis. At the beginning of the project, we also considered the dataset *Persons reporting a work-related health problem by type of diagnosis*. Although we eventually excluded it because too many values were marked as “low reliability”, we still believe that work-related well-being would be an interesting aspect to explore further, especially because European countries often differ culturally in their relationship with work and work-life balance.

We also identified additional factors that could enrich the analysis in future developments. For example, environmental and geographic variables such as sunlight exposure or climate differences could play an important role in mental well-being and mood, particularly when comparing Northern and Southern Europe.

Finally, the project currently focuses only on Europe. Extending the analysis to a global scale could provide a much broader perspective on the relationship between wealth and happiness. Including regions with very different economic conditions, such as parts of Africa or Asia, could reveal patterns that are not visible within Europe alone. However, this would also introduce additional challenges related to data quality, consistency, and availability.

## **7. Contributions**

**PAB 66**
- Monica Santinelli (s260228) -- data cleaning + geo data and maps + notebook
- Nico Schaul (s254419) -- data sources + data cleaning + barplots 
- Themistoklis Charmpis (s260236) -- data sources + website

## **8. References**

[1] Ranking of the happiest cities in the world: https://happy-city-index.com/       
[2] Eurostat database: https://ec.europa.eu/eurostat/data/database          
[3] Population dataset - Eurostat database: https://ec.europa.eu/eurostat/databrowser/view/tps00001/default/table?lang=en      
[4] Shapefile of European countries: https://data.dtu.dk/articles/dataset/Shapefile_of_European_countries/23686383          
[5] Shapefile of Austria: https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_AUT_0.json          